### Caso não tenha as libs instaladas no Kernel

In [1]:
%pip install plotly pandas scikit-learn opencv-python
%pip install --upgrade nbformat


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### Import das libs

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import math
import warnings
import glob
import os
from pathlib import Path

import cv2
from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


### Análise dos dados

In [3]:
lista_dfs = []
caminhos_ficheiros = glob.glob('../logs/*.csv') or glob.glob('logs/*.csv')

for caminho in caminhos_ficheiros:
    df_temp = pd.read_csv(caminho)
    
    df_temp['altitude'] = -df_temp['z']
    
    lista_dfs.append(df_temp)

#### Análise da trajetória

In [4]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_3d = go.Figure()

    # Adiciona a linha da Trajetória Real
    fig_3d.add_trace(go.Scatter3d(
        x=df['x'], y=df['y'], z=df['altitude'],
        mode='lines',
        line=dict(color='royalblue', width=4),
        name='Trajetória Real (Odometria)'
    ))

    # Adiciona o Ponto de Partida
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[0]], y=[df['y'].iloc[0]], z=[df['altitude'].iloc[0]],
        mode='markers',
        marker=dict(color='green', size=6),
        name='Ponto de Partida'
    ))

    # Adiciona o Ponto Final
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[-1]], y=[df['y'].iloc[-1]], z=[df['altitude'].iloc[-1]],
        mode='markers',
        marker=dict(color='red', size=6, symbol='x'),
        name='Ponto Final da Run'
    ))

    fig_3d.update_layout(
        title=f'Análise de Trajetória 3D do VANT - Run: {timestamp}',
        scene=dict(
            xaxis_title='Posição X (Metros)',
            yaxis_title='Posição Y (Metros)',
            zaxis_title='Posição Z / Altitude (Metros)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=0.5)) 
        ),
        legend=dict(x=0, y=1),
        margin=dict(l=0, r=0, b=0, t=40) 
    )

    fig_3d.show()

#### Análise do Pitch and Roll

In [5]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_2d = go.Figure()

    # Adiciona a linha de Roll
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['roll_speed'],
        mode='lines',
        name='Roll',
        opacity=0.7
    ))

    # Adiciona a linha de Pitch
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['pitch_speed'],
        mode='lines',
        name='Pitch',
        opacity=0.7
    ))

    fig_2d.update_layout(
        title=f'Esforço de Controle: Velocidades Angulares - Run: {timestamp}',
        xaxis_title='Tempo de Voo (Segundos)',
        yaxis_title='Velocidade Angular (rad/s)',
        template='plotly_white',
        hovermode='x unified' # Cria uma linha vertical interativa ao passar o mouse
    )

    fig_2d.show()

#### Analise do depth ground truth do Gazebo

Esta analise usa os pares RGB/depth salvos em `datasets/depth_ground_truth/run_*/metadata.csv` para transformar o depth renderizado pelo Gazebo em metricas por frame. A ideia e medir quando a cena ficou visualmente proxima da camera monocular e cruzar isso com IMU/atitude, deixando pronto o terreno para o treino futuro do modelo monocular.

Fontes Documentação: 
[Gazebo DepthCamera](https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html), 
[NumPy percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html), 
[Plotly multiple axes](https://plotly.com/python/multiple-axes/), 

Fontes Artigos: 
[RealTimeMonocular2022](https://doi.org/10.1109/TITS.2022.3160741), 
[Vyas2022](https://doi.org/10.48550/arXiv.2205.01399), 
[Tarrio2015](https://doi.org/10.1109/iccv.2015.87).

In [6]:
def localizar_raiz_projeto_depth():
    '''
    Localiza a raiz do projeto a partir do notebook aberto.

    O notebook pode ser executado a partir de estudos_e_analises/ ou da raiz do
    repositorio. A funcao procura a pasta datasets/depth_ground_truth nesses niveis
    para evitar caminhos absolutos presos ao Windows ou ao WSL.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [Gazebo DepthCamera] https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html
    '''

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / 'datasets' / 'depth_ground_truth').exists():
            return candidato
    return Path.cwd()


def localizar_ultima_run_depth(base_dir):
    '''
    Retorna o metadata.csv da run de depth ground truth mais recente.

    A pasta gerada pelo controlador segue o padrao run_<data_hora>. Usar a ultima run
    facilita reexecutar a analise logo depois de um voo sem editar o notebook.

    Fontes:
    [Python pathlib glob] https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
    [Pandas read_csv] https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
    '''

    depth_dir = base_dir / 'datasets' / 'depth_ground_truth'
    metadados = sorted(depth_dir.glob('run_*/metadata.csv'))
    return metadados[-1] if metadados else None


def resolver_arquivo_depth(depth_path, run_dir):
    '''
    Resolve o caminho do arquivo NPY mesmo quando o CSV veio do WSL.

    O metadata.csv pode armazenar caminhos como /home/prograf4080/...; quando eles nao
    existem no ambiente atual, a funcao reconstrui o caminho local usando run_dir/depth_m
    e o nome do arquivo.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [NumPy load] https://numpy.org/doc/stable/reference/generated/numpy.load.html
    '''

    caminho_original = Path(str(depth_path))
    if caminho_original.exists():
        return caminho_original
    return run_dir / 'depth_m' / caminho_original.name


def enriquecer_metadata_depth(df, run_dir, limiares=(2.0, 5.0, 10.0)):
    '''
    Calcula percentis e porcentagens de pixels proximos para cada mapa de depth.

    A funcao le os arquivos NPY em metros, descarta valores invalidos e mede P10/P50/P90
    alem da fracao de pixels abaixo de limiares de proximidade. Essas metricas convertem
    o depth do simulador em sinais mais simples para comparar com IMU, pan compensado e
    futuro treinamento monocular.

    Fontes:
    [NumPy percentile] https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
    [Artigo - RealTimeMonocular2022] https://doi.org/10.1109/TITS.2022.3160741
    [Artigo - Vyas2022] https://doi.org/10.48550/arXiv.2205.01399
    '''

    df = df.copy()
    if 'sample_id' in df.columns:
        df['sample_id'] = df['sample_id'].astype(str).str.zfill(6)

    metricas = {
        'depth_p10_m': [],
        'depth_p50_m': [],
        'depth_p90_m': [],
        'valid_px_pct': [],
    }
    for limiar in limiares:
        metricas[f'depth_close_{int(limiar)}m_pct'] = []

    for _, row in df.iterrows():
        arquivo_depth = resolver_arquivo_depth(row.get('depth_path', ''), run_dir)
        if not arquivo_depth.exists():
            for valores in metricas.values():
                valores.append(np.nan)
            continue

        depth = np.load(arquivo_depth, mmap_mode='r')
        validos = np.isfinite(depth) & (depth > 0.0)
        valores_depth = np.asarray(depth[validos], dtype=float)

        if valores_depth.size == 0:
            metricas['depth_p10_m'].append(np.nan)
            metricas['depth_p50_m'].append(np.nan)
            metricas['depth_p90_m'].append(np.nan)
            metricas['valid_px_pct'].append(0.0)
            for limiar in limiares:
                metricas[f'depth_close_{int(limiar)}m_pct'].append(np.nan)
            continue

        metricas['depth_p10_m'].append(float(np.percentile(valores_depth, 10)))
        metricas['depth_p50_m'].append(float(np.percentile(valores_depth, 50)))
        metricas['depth_p90_m'].append(float(np.percentile(valores_depth, 90)))
        metricas['valid_px_pct'].append(float(validos.mean() * 100.0))
        for limiar in limiares:
            metricas[f'depth_close_{int(limiar)}m_pct'].append(float((valores_depth < limiar).mean() * 100.0))

    for coluna, valores in metricas.items():
        df[coluna] = valores

    return df


raiz_projeto = localizar_raiz_projeto_depth()
metadata_depth = localizar_ultima_run_depth(raiz_projeto)

if metadata_depth is None:
    display(Markdown('Nenhuma run de depth encontrada em `datasets/depth_ground_truth`.'))
else:
    depth_df = pd.read_csv(metadata_depth)
    colunas_numericas = [
        'rgb_timestamp_s', 'depth_timestamp_s', 'depth_age_s',
        'x', 'y', 'z', 'roll', 'pitch', 'yaw',
        'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z',
        'depth_min_m', 'depth_mean_m', 'depth_max_m', 'pan_comp_delta_rad'
    ]
    for coluna in colunas_numericas:
        if coluna in depth_df.columns:
            depth_df[coluna] = pd.to_numeric(depth_df[coluna], errors='coerce')

    if 'pan_comp_delta_rad' not in depth_df.columns:
        depth_df['pan_comp_delta_rad'] = np.nan

    depth_df = depth_df.sort_values('rgb_timestamp_s').reset_index(drop=True)
    depth_df['tempo_s'] = depth_df['rgb_timestamp_s'] - depth_df['rgb_timestamp_s'].iloc[0]
    depth_df['gyro_norm'] = np.sqrt(depth_df['gyro_x']**2 + depth_df['gyro_y']**2 + depth_df['gyro_z']**2)
    depth_df['accel_norm'] = np.sqrt(depth_df['accel_x']**2 + depth_df['accel_y']**2 + depth_df['accel_z']**2)
    depth_df = enriquecer_metadata_depth(depth_df, metadata_depth.parent)

    print(f'Run analisada: {metadata_depth.parent.name}')
    print(f'Amostras RGB/depth: {len(depth_df)}')
    print(f'Desalinhamento medio RGB-depth: {depth_df["depth_age_s"].mean() * 1000:.1f} ms')

    resumo_depth = [
        'depth_age_s', 'depth_min_m', 'depth_mean_m', 'depth_p10_m', 'depth_p50_m',
        'depth_close_2m_pct', 'depth_close_5m_pct', 'depth_close_10m_pct',
        'gyro_norm', 'accel_norm', 'pan_comp_delta_rad'
    ]
    display(depth_df[[c for c in resumo_depth if c in depth_df.columns]].describe().T)

    fig_depth = go.Figure()
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_mean_m'], mode='lines', name='depth media'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_p10_m'], mode='lines', name='depth P10'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_p50_m'], mode='lines', name='depth P50'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_close_5m_pct'], mode='lines', name='pixels < 5 m (%)', yaxis='y2'))
    fig_depth.update_layout(
        title='Depth ground truth: distancia da cena e area proxima',
        xaxis_title='Tempo desde o primeiro par (s)',
        yaxis=dict(title='Profundidade (m)'),
        yaxis2=dict(title='Pixels < 5 m (%)', overlaying='y', side='right'),
        template='plotly_white',
        hovermode='x unified'
    )
    fig_depth.show()

    cor_pontos = depth_df['pan_comp_delta_rad'].abs()
    titulo_cor = '|pan compensado| (rad)'
    if cor_pontos.isna().all():
        cor_pontos = depth_df['depth_mean_m']
        titulo_cor = 'depth media (m)'

    fig_imu = go.Figure()
    fig_imu.add_trace(go.Scatter(
        x=depth_df['depth_close_5m_pct'],
        y=depth_df['gyro_norm'],
        mode='markers',
        marker=dict(size=8, color=cor_pontos, colorscale='Turbo', colorbar=dict(title=titulo_cor)),
        text=depth_df['sample_id'],
        customdata=np.stack([depth_df['tempo_s'], depth_df['depth_p10_m'], depth_df['accel_norm']], axis=-1),
        hovertemplate='amostra=%{text}<br>t=%{customdata[0]:.2f}s<br>pixels < 5m=%{x:.2f}%<br>gyro=%{y:.3f}rad/s<br>depth P10=%{customdata[1]:.2f}m<br>accel=%{customdata[2]:.2f}m/s2<extra></extra>'
    ))
    fig_imu.update_layout(
        title='Frames criticos: proximidade visual x giro do drone',
        xaxis_title='Pixels validos com depth < 5 m (%)',
        yaxis_title='Norma do giroscopio (rad/s)',
        template='plotly_white'
    )
    fig_imu.show()

    colunas_ranking = [
        'sample_id', 'tempo_s', 'depth_mean_m', 'depth_p10_m', 'depth_close_2m_pct',
        'depth_close_5m_pct', 'gyro_norm', 'accel_norm', 'pan_comp_delta_rad'
    ]
    ranking = depth_df.sort_values(['depth_close_5m_pct', 'gyro_norm'], ascending=False)
    display(Markdown('**Frames mais interessantes para inspecao/treino:**'))
    display(ranking[[c for c in colunas_ranking if c in ranking.columns]].head(12))

Run analisada: run_20260517_003517
Amostras RGB/depth: 44
Desalinhamento medio RGB-depth: 49.9 ms


,count,mean,std,min,25%,50%,75%,max
depth_age_s,44.0,0.049909,0.016849,0.032000,0.032000,0.064000,0.064000,0.068000
depth_min_m,44.0,0.258351,0.387801,0.100001,0.100013,0.118791,0.155244,1.517897
depth_mean_m,44.0,16.785077,3.877542,12.518493,13.413673,15.825388,19.242587,28.952608
depth_p10_m,44.0,1.671592,0.889887,0.268502,1.021565,1.914291,2.260962,3.906522
depth_p50_m,44.0,5.291103,3.316451,0.536424,1.967914,5.729702,7.218708,14.522176
depth_close_2m_pct,44.0,23.852782,24.882586,1.532311,6.474798,11.415437,48.576183,68.222417
depth_close_5m_pct,44.0,48.624541,16.278962,21.512579,37.299560,44.033497,69.138012,73.853632
depth_close_10m_pct,44.0,67.459943,9.220208,40.573284,62.714073,70.461644,74.523386,78.043351
gyro_norm,44.0,0.721067,0.622235,0.000627,0.144851,0.549846,1.041489,2.148035
accel_norm,44.0,10.015760,1.955542,0.121185,9.792419,10.161608,10.812382,12.291117


**Frames mais interessantes para inspecao/treino:**

,sample_id,tempo_s,depth_mean_m,depth_p10_m,depth_close_2m_pct,depth_close_5m_pct,gyro_norm,accel_norm,pan_comp_delta_rad
6,000007,2.936,14.873105,0.268502,68.222417,73.853632,0.000627,9.794542,-0.000004
5,000006,2.308,15.476609,0.281306,66.880508,72.717521,0.001350,9.790484,-0.000053
4,000005,1.748,15.748754,0.285216,66.257447,72.146998,0.001164,9.801261,-0.000007
7,000008,4.124,15.929793,0.289884,65.875812,71.846515,0.000953,9.785197,0.000005
0,000001,0.000,16.005783,0.290576,65.702114,71.686832,0.001081,9.790269,0.000013
1,000002,0.724,16.005739,0.290585,65.699282,71.683076,0.000849,9.803021,0.000016
2,000003,1.088,16.005735,0.290587,65.699553,71.682805,0.001271,9.793063,-0.000006
3,000004,1.352,16.009060,0.290623,65.693215,71.672681,0.001378,9.786960,0.000012
40,000041,31.448,15.035715,1.160006,45.032546,71.478431,0.303503,10.883903,0.003209
43,000044,33.428,13.037973,0.450441,60.028182,71.140127,0.079323,0.121185,-0.006056


#### Treino baseline com MLP simples para depth/risco

Esta celula implementa o baseline ate a etapa 5 do plano: monta um dataset supervisionado com RGB monocular, optical flow resumido e IMU; calcula alvos a partir do depth ground truth do Gazebo; separa treino/validacao/teste; treina uma MLP simples; e compara o erro contra um baseline de media. O objetivo aqui nao e prever um mapa denso perfeito, mas testar se uma MLP pequena consegue inferir indicadores uteis de profundidade/risco.

Fontes: [scikit-learn MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html), [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [OpenCV Farneback Optical Flow](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html), [OpenCV Canny](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html), [RealTimeMonocular2022](https://doi.org/10.1109/TITS.2022.3160741), [Vyas2022](https://doi.org/10.48550/arXiv.2205.01399), [Tarrio2015](https://doi.org/10.1109/iccv.2015.87).


In [7]:
import math
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def localizar_raiz_projeto_mlp():
    '''
    Localiza a raiz do repositorio para o treino MLP.

    O notebook pode ser executado a partir da raiz do projeto ou da pasta
    estudos_e_analises. A funcao procura datasets/depth_ground_truth nos ancestrais mais
    proximos para evitar caminhos absolutos presos ao Windows ou ao WSL.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [Gazebo DepthCamera] https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html
    '''

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / 'datasets' / 'depth_ground_truth').exists():
            return candidato
    return Path.cwd()


def listar_metadados_depth_mlp(base_dir):
    '''
    Lista as runs de depth ground truth disponiveis para treinamento.

    Cada metadata.csv define uma run supervisionada. O identificador da pasta run_* e usado
    como grupo de validacao para evitar misturar frames quase identicos entre treino e teste
    quando houver varias coletas.

    Fontes:
    [Python pathlib glob] https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
    [Pandas read_csv] https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
    '''

    return sorted((base_dir / 'datasets' / 'depth_ground_truth').glob('run_*/metadata.csv'))


def resolver_arquivo_da_run(valor_caminho, run_dir, subdir):
    '''
    Resolve caminhos absolutos do CSV ou reconstrui o caminho dentro da run local.

    Os metadados podem ter sido gravados no WSL com caminhos /home/prograf4080/..., mas a
    analise pode estar rodando no Windows. Quando o caminho original nao existe, a funcao
    usa apenas o nome do arquivo dentro de rgb/ ou depth_m/.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [NumPy load] https://numpy.org/doc/stable/reference/generated/numpy.load.html
    '''

    caminho = Path(str(valor_caminho))
    if caminho.exists():
        return caminho
    return run_dir / subdir / caminho.name


def carregar_rgb_mlp(row, run_dir):
    '''
    Carrega a imagem RGB/BGR associada a uma linha do metadata.csv.

    O OpenCV entrega a imagem em BGR. Essa representacao e mantida porque as features usam
    conversoes internas do proprio OpenCV para cinza e HSV.

    Fontes:
    [OpenCV imread] https://docs.opencv.org/4.x/d4/da8/group__imgcodecs.html
    [OpenCV color conversions] https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html
    '''

    caminho_rgb = resolver_arquivo_da_run(row.get('rgb_path', ''), run_dir, 'rgb')
    imagem = cv2.imread(str(caminho_rgb), cv2.IMREAD_COLOR)
    if imagem is None:
        raise FileNotFoundError(f'RGB nao encontrado ou invalido: {caminho_rgb}')
    return imagem


def calcular_alvos_depth_mlp(row, run_dir, limiares=(2.0, 5.0, 10.0)):
    '''
    Calcula os alvos supervisionados da MLP a partir do mapa de profundidade.

    Os alvos sao percentis de profundidade e porcentagens de pixels mais proximos que alguns
    limiares. Essa escolha transforma o mapa denso do Gazebo em sinais compactos, adequados
    para uma MLP simples.

    Fontes:
    [NumPy percentile] https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
    [Artigo - RealTimeMonocular2022] https://doi.org/10.1109/TITS.2022.3160741
    [Artigo - Vyas2022] https://doi.org/10.48550/arXiv.2205.01399
    '''

    caminho_depth = resolver_arquivo_da_run(row.get('depth_path', ''), run_dir, 'depth_m')
    depth = np.load(caminho_depth, mmap_mode='r')
    validos = np.isfinite(depth) & (depth > 0.0)
    valores = np.asarray(depth[validos], dtype=float)
    if valores.size == 0:
        raise ValueError(f'Depth sem pixels validos: {caminho_depth}')

    alvos = {
        'depth_p10_m': float(np.percentile(valores, 10)),
        'depth_p50_m': float(np.percentile(valores, 50)),
        'depth_p90_m': float(np.percentile(valores, 90)),
    }
    for limiar in limiares:
        alvos[f'depth_close_{int(limiar)}m_pct'] = float((valores < limiar).mean() * 100.0)
    return alvos


def extrair_features_visuais_mlp(imagem_bgr, gray_anterior=None, grid=(4, 3), tamanho=(96, 72)):
    '''
    Extrai features visuais compactas da imagem monocular e do optical flow.

    A imagem e reduzida para uma resolucao pequena, convertida para cinza e dividida em uma
    grade. Para cada bloco sao calculados media, desvio padrao, densidade de bordas, fluxo
    medio e fluxo radial medio. Isso cria uma entrada tabular simples o suficiente para MLP.

    Fontes:
    [OpenCV Canny] https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html
    [OpenCV Farneback Optical Flow] https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html
    [Artigo - Tarrio2015] https://doi.org/10.1109/iccv.2015.87
    '''

    imagem_pequena = cv2.resize(imagem_bgr, tamanho, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(imagem_pequena, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(imagem_pequena, cv2.COLOR_BGR2HSV)
    edges = cv2.Canny(gray, 60, 160)

    if gray_anterior is None:
        fluxo_mag = np.zeros_like(gray, dtype=float)
        fluxo_radial = np.zeros_like(gray, dtype=float)
    else:
        flow = cv2.calcOpticalFlowFarneback(
            gray_anterior,
            gray,
            None,
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0,
        )
        fluxo_mag = np.linalg.norm(flow, axis=2)
        yy, xx = np.mgrid[0:gray.shape[0], 0:gray.shape[1]]
        radial = np.stack([xx - gray.shape[1] * 0.5, yy - gray.shape[0] * 0.5], axis=2).astype(float)
        radial_norm = np.linalg.norm(radial, axis=2, keepdims=True) + 1e-6
        radial_unit = radial / radial_norm
        fluxo_radial = np.sum(flow * radial_unit, axis=2)

    features = {
        'gray_mean': float(gray.mean()),
        'gray_std': float(gray.std()),
        'edge_density': float((edges > 0).mean()),
        'sat_mean': float(hsv[:, :, 1].mean()),
        'sat_std': float(hsv[:, :, 1].std()),
        'flow_mag_mean': float(fluxo_mag.mean()),
        'flow_radial_mean': float(fluxo_radial.mean()),
    }

    hist = cv2.calcHist([gray], [0], None, [8], [0, 256]).flatten()
    hist = hist / max(float(hist.sum()), 1.0)
    for i, valor in enumerate(hist):
        features[f'gray_hist_{i}'] = float(valor)

    cols, rows = grid
    h, w = gray.shape
    for gy in range(rows):
        for gx in range(cols):
            y0, y1 = int(gy * h / rows), int((gy + 1) * h / rows)
            x0, x1 = int(gx * w / cols), int((gx + 1) * w / cols)
            prefix = f'g{gy}_{gx}'
            bloco_gray = gray[y0:y1, x0:x1]
            bloco_edges = edges[y0:y1, x0:x1]
            bloco_mag = fluxo_mag[y0:y1, x0:x1]
            bloco_radial = fluxo_radial[y0:y1, x0:x1]
            features[f'{prefix}_gray_mean'] = float(bloco_gray.mean())
            features[f'{prefix}_gray_std'] = float(bloco_gray.std())
            features[f'{prefix}_edge_density'] = float((bloco_edges > 0).mean())
            features[f'{prefix}_flow_mag'] = float(bloco_mag.mean())
            features[f'{prefix}_flow_radial'] = float(bloco_radial.mean())

    return features, gray


def extrair_features_estado_mlp(row):
    '''
    Extrai features escalares de atitude, IMU e compensacao visual.

    As features nao usam diretamente depth_min/depth_mean/depth_max para evitar vazamento do
    alvo. Elas usam apenas informacao que a camera monocular/estado do drone poderia ter no
    momento do frame.

    Fontes:
    [PX4 SensorCombined] https://docs.px4.io/main/en/msg_docs/SensorCombined.html
    [PX4 VehicleAttitude] https://docs.px4.io/main/en/msg_docs/VehicleAttitude
    '''

    colunas = [
        'roll', 'pitch', 'yaw',
        'gyro_x', 'gyro_y', 'gyro_z',
        'accel_x', 'accel_y', 'accel_z',
        'pan_comp_delta_rad',
    ]
    features = {}
    for coluna in colunas:
        valor = pd.to_numeric(row.get(coluna, 0.0), errors='coerce')
        features[coluna] = 0.0 if pd.isna(valor) else float(valor)

    features['gyro_norm'] = math.sqrt(features['gyro_x']**2 + features['gyro_y']**2 + features['gyro_z']**2)
    features['accel_norm'] = math.sqrt(features['accel_x']**2 + features['accel_y']**2 + features['accel_z']**2)
    features['tilt_abs'] = abs(features['roll']) + abs(features['pitch'])
    return features


def montar_dataset_mlp_depth(base_dir):
    '''
    Monta o dataset tabular usado pelo baseline MLP.

    A funcao percorre todas as runs de depth, extrai features de RGB, optical flow e IMU, e
    calcula os alvos compactos de profundidade. Cada amostra preserva run_id e sample_id para
    separacao temporal ou por run.

    Fontes:
    [Pandas DataFrame] https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
    [OpenCV Optical Flow] https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html
    [scikit-learn supervised learning] https://scikit-learn.org/stable/supervised_learning.html
    '''

    linhas_x = []
    linhas_y = []
    linhas_meta = []
    metadados = listar_metadados_depth_mlp(base_dir)

    for metadata_path in metadados:
        run_dir = metadata_path.parent
        run_id = run_dir.name.replace('run_', '')
        df = pd.read_csv(metadata_path).sort_values('rgb_timestamp_s').reset_index(drop=True)
        gray_anterior = None

        for _, row in df.iterrows():
            try:
                imagem = carregar_rgb_mlp(row, run_dir)
                features_visuais, gray_atual = extrair_features_visuais_mlp(imagem, gray_anterior)
                features_estado = extrair_features_estado_mlp(row)
                alvos = calcular_alvos_depth_mlp(row, run_dir)
            except Exception as exc:
                print(f'Amostra ignorada em {run_id}: {exc}')
                continue

            gray_anterior = gray_atual
            linhas_x.append({**features_visuais, **features_estado})
            linhas_y.append(alvos)
            linhas_meta.append({
                'run_id': run_id,
                'sample_id': str(row.get('sample_id', '')).zfill(6),
                'rgb_timestamp_s': float(pd.to_numeric(row.get('rgb_timestamp_s', np.nan), errors='coerce')),
            })

    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


def separar_treino_validacao_teste_mlp(meta_df, random_state=42):
    '''
    Separa indices de treino, validacao e teste sem embaralhar frames de uma mesma run.

    Quando existem tres ou mais runs, a separacao e feita por run_id. Com uma unica run, a
    funcao usa uma divisao temporal 70/15/15, que e menos forte que validacao por run, mas
    evita misturar frames futuros no treino.

    Fontes:
    [scikit-learn model evaluation] https://scikit-learn.org/stable/model_selection.html
    [NumPy random generator] https://numpy.org/doc/stable/reference/random/generator.html
    '''

    n = len(meta_df)
    indices = np.arange(n)
    runs = meta_df['run_id'].dropna().unique()

    if len(runs) >= 3:
        rng = np.random.default_rng(random_state)
        runs = np.array(runs)
        rng.shuffle(runs)
        n_train = max(1, int(len(runs) * 0.7))
        n_val = max(1, int(len(runs) * 0.15))
        train_runs = set(runs[:n_train])
        val_runs = set(runs[n_train:n_train + n_val])
        test_runs = set(runs[n_train + n_val:])
        if not test_runs:
            test_runs = {runs[-1]}
            train_runs.discard(runs[-1])
        split = {
            'train': indices[meta_df['run_id'].isin(train_runs).to_numpy()],
            'val': indices[meta_df['run_id'].isin(val_runs).to_numpy()],
            'test': indices[meta_df['run_id'].isin(test_runs).to_numpy()],
            'modo': 'por run_id',
        }
    else:
        n_train = max(1, int(n * 0.70))
        n_val = max(1, int(n * 0.15))
        split = {
            'train': indices[:n_train],
            'val': indices[n_train:n_train + n_val],
            'test': indices[n_train + n_val:],
            'modo': 'temporal dentro da unica run disponivel',
        }

    if len(split['test']) == 0:
        split['test'] = split['val']
    if len(split['val']) == 0:
        split['val'] = split['test']
    return split


def avaliar_predicoes_mlp(y_true, y_pred, targets, nome_modelo, split_name):
    '''
    Calcula MAE, RMSE e correlacao por alvo para um conjunto de predicoes.

    A correlacao e informativa para saber se o modelo acompanha a tendencia dos frames, mesmo
    quando a escala absoluta ainda tem erro. MAE e RMSE indicam o erro direto das metricas de
    profundidade/risco.

    Fontes:
    [scikit-learn metrics] https://scikit-learn.org/stable/modules/model_evaluation.html
    [NumPy corrcoef] https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html
    '''

    linhas = []
    for idx, alvo in enumerate(targets):
        real = np.asarray(y_true[:, idx], dtype=float)
        pred = np.asarray(y_pred[:, idx], dtype=float)
        corr = np.nan
        if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9:
            corr = float(np.corrcoef(real, pred)[0, 1])
        linhas.append({
            'modelo': nome_modelo,
            'split': split_name,
            'alvo': alvo,
            'MAE': float(mean_absolute_error(real, pred)),
            'RMSE': float(mean_squared_error(real, pred) ** 0.5),
            'corr': corr,
        })
    return linhas


def treinar_avaliar_mlp_depth(X, y, meta_df, random_state=42):
    '''
    Treina a MLP simples e compara contra um baseline de media.

    A MLP recebe features normalizadas e alvos normalizados por StandardScaler. O baseline
    DummyRegressor prev? a media do treino, servindo como referencia minima: a MLP so e util
    se reduzir erro e/ou aumentar correlacao contra esse modelo burro.

    Fontes:
    [scikit-learn MLPRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
    [scikit-learn TransformedTargetRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html
    [scikit-learn DummyRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html
    '''

    targets = list(y.columns)
    split = separar_treino_validacao_teste_mlp(meta_df, random_state=random_state)
    X_values = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    y_values = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

    train_idx = split['train']
    val_idx = split['val']
    test_idx = split['test']

    mlp = TransformedTargetRegressor(
        regressor=Pipeline([
            ('x_scaler', StandardScaler()),
            ('mlp', MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation='relu',
                solver='lbfgs',
                alpha=0.01,
                max_iter=2000,
                random_state=random_state,
            )),
        ]),
        transformer=StandardScaler(),
    )
    dummy = DummyRegressor(strategy='mean')

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mlp.fit(X_values[train_idx], y_values[train_idx])
    dummy.fit(X_values[train_idx], y_values[train_idx])

    linhas = []
    predicoes = {}
    for split_name, idx in [('validacao', val_idx), ('teste', test_idx)]:
        pred_mlp = mlp.predict(X_values[idx])
        pred_dummy = dummy.predict(X_values[idx])
        predicoes[split_name] = {
            'idx': idx,
            'real': y_values[idx],
            'mlp': pred_mlp,
            'dummy': pred_dummy,
        }
        linhas.extend(avaliar_predicoes_mlp(y_values[idx], pred_mlp, targets, 'MLP', split_name))
        linhas.extend(avaliar_predicoes_mlp(y_values[idx], pred_dummy, targets, 'Media treino', split_name))

    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_mlp()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_depth(raiz_mlp)

if len(X_mlp) < 12:
    display(Markdown('Dataset insuficiente para treinar a MLP. Colete mais pares RGB/depth.'))
else:
    modelo_mlp_depth, baseline_media_depth, split_mlp, resultados_mlp_depth, predicoes_mlp_depth = treinar_avaliar_mlp_depth(
        X_mlp,
        y_mlp,
        meta_mlp,
    )

    display(Markdown(
        f'**Dataset MLP:** {len(X_mlp)} amostras, {X_mlp.shape[1]} features, '
        f'{y_mlp.shape[1]} alvos. Separacao usada: `{split_mlp["modo"]}`. '
        f'Treino={len(split_mlp["train"])}, validacao={len(split_mlp["val"])}, teste={len(split_mlp["test"])}.'
    ))

    display(resultados_mlp_depth.sort_values(['split', 'alvo', 'modelo']).reset_index(drop=True))

    resultados_teste = resultados_mlp_depth[resultados_mlp_depth['split'] == 'teste']
    fig_mae = go.Figure()
    for modelo in resultados_teste['modelo'].unique():
        parte = resultados_teste[resultados_teste['modelo'] == modelo]
        fig_mae.add_trace(go.Bar(x=parte['alvo'], y=parte['MAE'], name=modelo))
    fig_mae.update_layout(
        title='Erro MAE no teste: MLP x baseline de media',
        xaxis_title='Alvo supervisionado',
        yaxis_title='MAE',
        barmode='group',
        template='plotly_white',
    )
    fig_mae.show()

    targets = list(y_mlp.columns)
    pred_teste = predicoes_mlp_depth['teste']
    meta_teste = meta_mlp.iloc[pred_teste['idx']].reset_index(drop=True)
    alvo_plot = 'depth_close_5m_pct' if 'depth_close_5m_pct' in targets else targets[0]
    alvo_idx = targets.index(alvo_plot)

    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['real'][:, alvo_idx],
        mode='lines+markers',
        name='real',
    ))
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['mlp'][:, alvo_idx],
        mode='lines+markers',
        name='MLP',
    ))
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['dummy'][:, alvo_idx],
        mode='lines',
        name='media treino',
    ))
    fig_pred.update_layout(
        title=f'Predicao no teste para {alvo_plot}',
        xaxis_title='Amostras de teste em ordem temporal',
        yaxis_title=alvo_plot,
        template='plotly_white',
        hovermode='x unified',
    )
    fig_pred.show()

    tabela_predicoes = pd.DataFrame({
        'run_id': meta_teste['run_id'],
        'sample_id': meta_teste['sample_id'],
        f'{alvo_plot}_real': pred_teste['real'][:, alvo_idx],
        f'{alvo_plot}_mlp': pred_teste['mlp'][:, alvo_idx],
        f'{alvo_plot}_baseline_media': pred_teste['dummy'][:, alvo_idx],
    })
    display(Markdown('**Amostras de teste para inspecao:**'))
    display(tabela_predicoes.head(15))


**Dataset MLP:** 156 amostras, 88 features, 6 alvos. Separacao usada: `por run_id`. Treino=88, validacao=68, teste=68.

,modelo,split,alvo,MAE,RMSE,corr
0,MLP,teste,depth_close_10m_pct,6.056365,7.549357,0.723122
1,Media treino,teste,depth_close_10m_pct,8.658048,10.819424,NaN
2,MLP,teste,depth_close_2m_pct,9.784854,13.878973,0.862546
3,Media treino,teste,depth_close_2m_pct,20.932785,26.709275,NaN
4,MLP,teste,depth_close_5m_pct,9.195707,11.325337,0.811895
5,Media treino,teste,depth_close_5m_pct,14.760381,18.470189,NaN
6,MLP,teste,depth_p10_m,0.482706,0.601310,0.739294
7,Media treino,teste,depth_p10_m,0.641987,0.825083,NaN
8,MLP,teste,depth_p50_m,1.296248,1.641300,0.851885
9,Media treino,teste,depth_p50_m,2.357435,2.976901,NaN


**Amostras de teste para inspecao:**

,run_id,sample_id,depth_close_5m_pct_real,depth_close_5m_pct_mlp,depth_close_5m_pct_baseline_media
0,20260516_201713,000001,77.910156,72.804213,47.159054
1,20260516_201713,000002,77.910971,72.812137,47.159054
2,20260516_201713,000003,77.909928,72.809870,47.159054
3,20260516_201713,000004,77.906987,72.802452,47.159054
4,20260516_201713,000005,77.378924,72.114367,47.159054
5,20260516_201713,000006,77.157022,72.563981,47.159054
6,20260516_201713,000007,77.313489,72.579662,47.159054
7,20260516_201713,000008,78.491293,72.518747,47.159054
8,20260516_201713,000009,81.710941,72.601092,47.159054
9,20260516_201713,000010,95.068307,68.613805,47.159054


#### CNN simples com uma camada de aten??o para depth ground truth

Esta se??o implementa o pr?ximo baseline em PyTorch: uma CNN encoder-decoder pequena que estima um mapa de depth reduzido a partir da imagem monocular RGB salva no dataset do Gazebo. A varia??o principal injeta uma ?nica camada de self-attention espacial no gargalo da CNN e compara o resultado contra a mesma CNN sem aten??o.

O alvo continua sendo o depth ground truth do simulador, n?o uma depth camera real. Para evitar que valores muito longos dominem o treino, o depth ? limitado a 50 m e treinado em `log1p(depth)`. O mapa previsto ? reduzido para 30x40 pixels, suficiente para validar se existe sinal visual ?til antes de pensar em uma arquitetura maior.

Fontes usadas como refer?ncia de implementa??o: [PyTorch Conv2d](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Conv2d.html), [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html), [TensorFlow Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D), [TensorFlow MultiHeadAttention](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention), [Attention Is All You Need](https://arxiv.org/abs/1706.03762) e [Depth Map Prediction from a Single Image using a Multi-Scale Deep Network](https://arxiv.org/abs/1406.2283).


In [ ]:
import math
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset, Subset
except ImportError as exc:
    raise ImportError(
        "Esta celula precisa do PyTorch. Instale com: "
        "python3 -m pip install -r requirements.txt"
    ) from exc


PROJETO_RAIZ_CNN = Path.cwd()
DATASET_DEPTH_CNN = PROJETO_RAIZ_CNN / "datasets" / "depth_ground_truth"
INPUT_SIZE_CNN = (120, 160)  # altura, largura
DEPTH_TARGET_SIZE_CNN = (30, 40)  # altura, largura
DEPTH_MAX_TREINO_M = 50.0
BATCH_SIZE_CNN = 8
EPOCHS_CNN_DEPTH = 6
LR_CNN_DEPTH = 1e-3


def localizar_raiz_projeto_cnn(nome_dataset="datasets"):
    '''Localiza a raiz do projeto para a analise CNN de depth.

    A busca parte do diretorio atual do notebook e sobe na arvore de pastas ate
    encontrar a pasta de datasets usada pelas coletas de depth ground truth.

    Args:
        nome_dataset: Nome da pasta que identifica a raiz do projeto.

    Returns:
        Path absoluto para a raiz do projeto.

    Raises:
        FileNotFoundError: Se a pasta de datasets nao for encontrada.
    '''
    atual = Path.cwd().resolve()
    candidatos = [atual, *atual.parents]

    for candidato in candidatos:
        if (candidato / nome_dataset).exists():
            return candidato

    raise FileNotFoundError(
        "Nao encontrei a raiz do projeto a partir do diretorio atual do notebook."
    )


def resolver_caminho_depth_cnn(valor, raiz_projeto):
    '''Resolve caminhos de RGB/depth gravados no metadata.csv.

    O metadata pode conter caminhos absolutos do WSL ou caminhos relativos. Esta
    funcao tenta reapontar esses caminhos para a copia local do projeto antes de
    descartar a amostra.

    Args:
        valor: Caminho lido do CSV.
        raiz_projeto: Raiz local do projeto.

    Returns:
        Path resolvido quando o arquivo existe, ou None quando nao foi encontrado.
    '''
    if pd.isna(valor):
        return None

    caminho = Path(str(valor))

    if caminho.exists():
        return caminho

    texto = str(valor).replace("\\", "/")
    marcador = "/TCC_Drone/"

    if marcador in texto:
        relativo = texto.split(marcador, 1)[1]
        candidato = raiz_projeto / relativo
        if candidato.exists():
            return candidato

    candidato = raiz_projeto / texto
    if candidato.exists():
        return candidato

    return None


def coletar_amostras_depth_cnn(dataset_dir, raiz_projeto):
    '''Coleta pares RGB/depth disponiveis nas runs de depth ground truth.

    Cada linha valida aponta para um arquivo RGB e um arquivo `.npy` de depth do
    Gazebo. O identificador da run e preservado para permitir split por voo.

    Args:
        dataset_dir: Pasta `datasets/depth_ground_truth`.
        raiz_projeto: Raiz local do projeto.

    Returns:
        DataFrame com as colunas originais do metadata e caminhos resolvidos.
    '''
    linhas = []

    for metadata_path in sorted(Path(dataset_dir).glob("run_*/metadata.csv")):
        df = pd.read_csv(metadata_path)
        run_id = metadata_path.parent.name

        for _, row in df.iterrows():
            rgb_path = resolver_caminho_depth_cnn(row.get("rgb_path"), raiz_projeto)
            depth_path = resolver_caminho_depth_cnn(row.get("depth_path"), raiz_projeto)

            if rgb_path is None or depth_path is None:
                continue

            item = row.to_dict()
            item["run_id"] = run_id
            item["metadata_path"] = str(metadata_path)
            item["rgb_path_resolvido"] = str(rgb_path)
            item["depth_path_resolvido"] = str(depth_path)
            linhas.append(item)

    return pd.DataFrame(linhas)


def dividir_amostras_por_run(amostras):
    '''Divide amostras em treino, validacao e teste.

    Quando ha tres ou mais runs, a divisao e feita por run para reduzir vazamento
    temporal entre treino e avaliacao. Com menos runs, usa uma divisao temporal
    70/15/15 como fallback.

    Args:
        amostras: DataFrame retornado por `coletar_amostras_depth_cnn`.

    Returns:
        Tupla com listas de indices `(train_idx, val_idx, test_idx, descricao)`.
    '''
    if amostras.empty:
        return [], [], [], "sem amostras"

    run_ids = sorted(amostras["run_id"].dropna().unique())

    if len(run_ids) >= 3:
        test_run = run_ids[-1]
        val_run = run_ids[-2]
        train_runs = set(run_ids[:-2])

        train_idx = amostras.index[amostras["run_id"].isin(train_runs)].tolist()
        val_idx = amostras.index[amostras["run_id"].eq(val_run)].tolist()
        test_idx = amostras.index[amostras["run_id"].eq(test_run)].tolist()
        descricao = f"split por run: treino={sorted(train_runs)}, val={val_run}, teste={test_run}"
        return train_idx, val_idx, test_idx, descricao

    n = len(amostras)
    train_end = max(1, int(n * 0.70))
    val_end = max(train_end + 1, int(n * 0.85)) if n >= 3 else train_end
    indices = amostras.index.tolist()
    descricao = "split temporal 70/15/15 por haver menos de tres runs"
    return indices[:train_end], indices[train_end:val_end], indices[val_end:], descricao


def preparar_rgb_tensor(rgb_path, input_size):
    '''Carrega e normaliza uma imagem RGB para entrada da CNN.

    Args:
        rgb_path: Caminho da imagem RGB salva na coleta.
        input_size: Tupla `(altura, largura)` da entrada do modelo.

    Returns:
        Tensor float32 no formato `C x H x W`, normalizado para aproximadamente
        o intervalo [-1, 1].
    '''
    altura, largura = input_size
    imagem = Image.open(rgb_path).convert("RGB").resize((largura, altura), Image.BILINEAR)
    array = np.asarray(imagem, dtype=np.float32) / 255.0
    array = (array - 0.5) / 0.5
    array = np.transpose(array, (2, 0, 1))
    return torch.from_numpy(array.astype(np.float32))


def preparar_depth_tensor(depth_path, target_size, depth_max_m):
    '''Carrega o depth do Gazebo e prepara alvo reduzido em log-depth.

    Args:
        depth_path: Caminho do arquivo `.npy` com depth em metros.
        target_size: Tupla `(altura, largura)` do mapa-alvo reduzido.
        depth_max_m: Valor maximo usado no clipping do depth.

    Returns:
        Tupla `(depth_log, valid_mask, metricas)` onde `depth_log` e o alvo em
        `log1p(metros)`, `valid_mask` indica pixels validos e `metricas` resume
        profundidade proxima/mediana e percentual abaixo de 5 m.
    '''
    altura, largura = target_size
    depth = np.load(depth_path).astype(np.float32)
    valid_mask = np.isfinite(depth) & (depth > 0.0)

    depth_clip = np.where(valid_mask, np.clip(depth, 0.1, depth_max_m), depth_max_m)
    depth_small = cv2.resize(depth_clip, (largura, altura), interpolation=cv2.INTER_AREA)
    mask_small = cv2.resize(valid_mask.astype(np.float32), (largura, altura), interpolation=cv2.INTER_AREA)
    mask_small = (mask_small > 0.5).astype(np.float32)

    valid_values = depth_clip[valid_mask]
    if valid_values.size:
        p10 = float(np.percentile(valid_values, 10))
        p50 = float(np.percentile(valid_values, 50))
        close_5m_pct = float((valid_values < 5.0).mean() * 100.0)
    else:
        p10 = depth_max_m
        p50 = depth_max_m
        close_5m_pct = 0.0

    depth_log = np.log1p(depth_small).astype(np.float32)
    metricas = np.array([np.log1p(p10), np.log1p(p50), close_5m_pct / 100.0], dtype=np.float32)

    return (
        torch.from_numpy(depth_log[None, :, :]),
        torch.from_numpy(mask_small[None, :, :]),
        torch.from_numpy(metricas),
    )


class DepthGroundTruthDataset(Dataset):
    '''Dataset PyTorch para os pares RGB/depth ground truth do Gazebo.

    O dataset retorna a imagem monocular RGB, o mapa de depth reduzido em
    log-depth, uma mascara de pixels validos e metricas auxiliares para ajudar a
    avaliar o comportamento do modelo.
    '''

    def __init__(self, amostras, input_size=(120, 160), target_size=(30, 40), depth_max_m=50.0):
        '''Inicializa o dataset de depth ground truth.

        Args:
            amostras: DataFrame com caminhos resolvidos para RGB e depth.
            input_size: Tamanho `(altura, largura)` da imagem de entrada.
            target_size: Tamanho `(altura, largura)` do mapa de depth alvo.
            depth_max_m: Valor maximo usado para clipping do depth.
        '''
        self.amostras = amostras.reset_index(drop=True)
        self.input_size = input_size
        self.target_size = target_size
        self.depth_max_m = depth_max_m

    def __len__(self):
        '''Retorna a quantidade de amostras disponiveis no dataset.'''
        return len(self.amostras)

    def __getitem__(self, idx):
        '''Carrega uma amostra RGB/depth pelo indice informado.

        Args:
            idx: Indice inteiro da amostra.

        Returns:
            Dicionario com tensores `rgb`, `depth_log`, `mask`, `metrics` e
            metadados textuais usados para analise.
        '''
        row = self.amostras.iloc[idx]
        rgb = preparar_rgb_tensor(row["rgb_path_resolvido"], self.input_size)
        depth_log, mask, metrics = preparar_depth_tensor(
            row["depth_path_resolvido"], self.target_size, self.depth_max_m
        )

        return {
            "rgb": rgb,
            "depth_log": depth_log,
            "mask": mask,
            "metrics": metrics,
            "run_id": row["run_id"],
            "sample_id": int(row.get("sample_id", idx)),
            "rgb_path": row["rgb_path_resolvido"],
            "depth_path": row["depth_path_resolvido"],
        }


class ConvBlock(nn.Module):
    '''Bloco convolucional pequeno usado no encoder e decoder da CNN.'''

    def __init__(self, in_channels, out_channels, stride=1):
        '''Cria uma sequencia Conv2d, BatchNorm2d e ReLU.

        Args:
            in_channels: Numero de canais de entrada.
            out_channels: Numero de canais de saida.
            stride: Passo da convolucao, usado para reduzir resolucao no encoder.
        '''
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        '''Executa o bloco convolucional sobre o tensor de entrada.'''
        return self.block(x)


class SpatialSelfAttention2d(nn.Module):
    '''Camada unica de self-attention espacial para o gargalo da CNN.

    A camada converte o feature map `B x C x H x W` em uma sequencia de tokens
    `B x (H*W) x C`, aplica MultiheadAttention e reconstr?i o formato espacial.
    Um ganho residual inicializado em zero evita que a atencao perturbe demais a
    CNN no inicio do treinamento.
    '''

    def __init__(self, channels, num_heads=4, dropout=0.05):
        '''Inicializa a self-attention espacial.

        Args:
            channels: Dimensao dos tokens, igual ao numero de canais do feature map.
            num_heads: Quantidade de cabecas de atencao.
            dropout: Dropout aplicado na atencao durante o treino.
        '''
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        '''Aplica self-attention espacial preservando o formato `B x C x H x W`.'''
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        tokens_norm = self.norm(tokens)
        attended, _ = self.attn(tokens_norm, tokens_norm, tokens_norm, need_weights=False)
        tokens = tokens + self.gamma * attended
        return tokens.transpose(1, 2).reshape(batch, channels, height, width)


class DepthCNNBaseline(nn.Module):
    '''CNN simples para estimar depth monocular usando ground truth do Gazebo.

    O modelo possui um encoder convolucional, um gargalo opcional com uma unica
    camada de atencao espacial e duas cabecas: uma para mapa de depth reduzido e
    outra para metricas auxiliares de risco/profundidade.
    '''

    def __init__(self, usar_atencao=False):
        '''Cria a CNN baseline com ou sem a camada de atencao.

        Args:
            usar_atencao: Quando True, injeta `SpatialSelfAttention2d` no gargalo.
        '''
        super().__init__()
        self.usar_atencao = usar_atencao
        self.encoder = nn.Sequential(
            ConvBlock(3, 16, stride=2),
            ConvBlock(16, 32, stride=2),
            ConvBlock(32, 64, stride=2),
        )
        self.attention = SpatialSelfAttention2d(64, num_heads=4) if usar_atencao else nn.Identity()
        self.decoder = nn.Sequential(
            ConvBlock(64, 64),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(64, 32),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Softplus(),
        )
        self.metric_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 3),
        )

    def forward(self, x):
        '''Calcula o mapa de depth e as metricas auxiliares para um batch RGB.'''
        features = self.encoder(x)
        features = self.attention(features)
        depth_log = self.decoder(features)
        metrics = self.metric_head(features)
        return depth_log, metrics


def masked_smooth_l1_loss(pred, target, mask):
    '''Calcula Smooth L1 apenas nos pixels validos do mapa de depth.

    Args:
        pred: Mapa previsto em log-depth.
        target: Mapa alvo em log-depth.
        mask: Mascara binaria de pixels validos.

    Returns:
        Tensor escalar com a perda media nos pixels validos.
    '''
    loss = F.smooth_l1_loss(pred, target, reduction="none") * mask
    return loss.sum() / mask.sum().clamp_min(1.0)


def treinar_uma_epoca_depth_cnn(modelo, loader, optimizer, device):
    '''Treina a CNN por uma epoca.

    Args:
        modelo: Instancia de `DepthCNNBaseline`.
        loader: DataLoader de treino.
        optimizer: Otimizador PyTorch.
        device: Dispositivo de execucao.

    Returns:
        Perda media da epoca.
    '''
    modelo.train()
    perdas = []

    for batch in loader:
        rgb = batch["rgb"].to(device)
        depth_log = batch["depth_log"].to(device)
        mask = batch["mask"].to(device)
        metrics = batch["metrics"].to(device)

        optimizer.zero_grad(set_to_none=True)
        pred_depth_log, pred_metrics = modelo(rgb)
        map_loss = masked_smooth_l1_loss(pred_depth_log, depth_log, mask)
        metric_loss = F.smooth_l1_loss(pred_metrics, metrics)
        loss = map_loss + 0.2 * metric_loss
        loss.backward()
        optimizer.step()
        perdas.append(float(loss.detach().cpu()))

    return float(np.mean(perdas)) if perdas else math.nan


def avaliar_depth_cnn(modelo, loader, device, depth_max_m=50.0):
    '''Avalia a CNN em mapa de depth e metricas derivadas.

    Args:
        modelo: Instancia treinada de `DepthCNNBaseline`.
        loader: DataLoader de validacao ou teste.
        device: Dispositivo de execucao.
        depth_max_m: Limite maximo usado no treino, em metros.

    Returns:
        Dicionario com MAE/RMSE do mapa, erro de profundidade proxima/mediana,
        erro de percentual abaixo de 5 m e tempo medio de inferencia.
    '''
    modelo.eval()
    map_abs = []
    map_sq = []
    p10_abs = []
    p50_abs = []
    close5_abs = []
    infer_times = []

    with torch.no_grad():
        for batch in loader:
            rgb = batch["rgb"].to(device)
            depth_log = batch["depth_log"].to(device)
            mask = batch["mask"].to(device)

            start = time.perf_counter()
            pred_depth_log, _ = modelo(rgb)
            if device.type == "cuda":
                torch.cuda.synchronize()
            infer_times.append((time.perf_counter() - start) / max(1, rgb.shape[0]))

            pred_m = torch.expm1(pred_depth_log).clamp(0.0, depth_max_m)
            target_m = torch.expm1(depth_log).clamp(0.0, depth_max_m)
            diff = (pred_m - target_m) * mask
            valid_count = mask.sum().clamp_min(1.0)

            map_abs.append(float(diff.abs().sum().cpu() / valid_count.cpu()))
            map_sq.append(float(((diff ** 2).sum().cpu() / valid_count.cpu())))

            pred_flat = pred_m.flatten(1)
            target_flat = target_m.flatten(1)
            p10_abs.append(float((torch.quantile(pred_flat, 0.10, dim=1) - torch.quantile(target_flat, 0.10, dim=1)).abs().mean().cpu()))
            p50_abs.append(float((torch.quantile(pred_flat, 0.50, dim=1) - torch.quantile(target_flat, 0.50, dim=1)).abs().mean().cpu()))
            close_pred = (pred_flat < 5.0).float().mean(dim=1) * 100.0
            close_target = (target_flat < 5.0).float().mean(dim=1) * 100.0
            close5_abs.append(float((close_pred - close_target).abs().mean().cpu()))

    return {
        "map_mae_m": float(np.mean(map_abs)) if map_abs else math.nan,
        "map_rmse_m": float(np.sqrt(np.mean(map_sq))) if map_sq else math.nan,
        "p10_mae_m": float(np.mean(p10_abs)) if p10_abs else math.nan,
        "p50_mae_m": float(np.mean(p50_abs)) if p50_abs else math.nan,
        "close5_mae_pp": float(np.mean(close5_abs)) if close5_abs else math.nan,
        "inferencia_ms_por_frame": float(np.mean(infer_times) * 1000.0) if infer_times else math.nan,
    }


def treinar_modelo_depth_cnn(nome, usar_atencao, train_loader, val_loader, test_loader, device):
    '''Treina e avalia uma variante da CNN de depth.

    Args:
        nome: Nome exibido nos resultados.
        usar_atencao: Define se a variante usa self-attention no gargalo.
        train_loader: DataLoader de treino.
        val_loader: DataLoader de validacao.
        test_loader: DataLoader de teste.
        device: Dispositivo PyTorch.

    Returns:
        Tupla `(modelo, historico, metricas_teste)`.
    '''
    torch.manual_seed(42)
    modelo = DepthCNNBaseline(usar_atencao=usar_atencao).to(device)
    optimizer = torch.optim.AdamW(modelo.parameters(), lr=LR_CNN_DEPTH, weight_decay=1e-4)
    historico = []

    for epoch in range(1, EPOCHS_CNN_DEPTH + 1):
        train_loss = treinar_uma_epoca_depth_cnn(modelo, train_loader, optimizer, device)
        val_metrics = avaliar_depth_cnn(modelo, val_loader, device)
        historico.append({
            "modelo": nome,
            "epoch": epoch,
            "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })
        print(
            f"{nome} | epoca {epoch:02d}/{EPOCHS_CNN_DEPTH} | "
            f"loss={train_loss:.4f} | val_map_mae={val_metrics['map_mae_m']:.3f} m"
        )

    test_metrics = avaliar_depth_cnn(modelo, test_loader, device)
    return modelo, pd.DataFrame(historico), {"modelo": nome, **test_metrics}


def prever_um_exemplo_depth_cnn(modelo, dataset, indice, device):
    '''Gera predicao de depth para uma amostra do dataset.

    Args:
        modelo: Modelo treinado.
        dataset: Dataset completo usado nas analises.
        indice: Indice da amostra dentro do dataset.
        device: Dispositivo PyTorch.

    Returns:
        Dicionario com imagem RGB, mapa alvo, mapa previsto e metadados.
    '''
    modelo.eval()
    sample = dataset[indice]

    with torch.no_grad():
        pred_log, _ = modelo(sample["rgb"].unsqueeze(0).to(device))

    pred_m = torch.expm1(pred_log.squeeze(0).squeeze(0)).cpu().numpy()
    target_m = torch.expm1(sample["depth_log"].squeeze(0)).cpu().numpy()
    rgb = np.asarray(Image.open(sample["rgb_path"]).convert("RGB").resize((160, 120), Image.BILINEAR))

    return {
        "rgb": rgb,
        "target_m": target_m,
        "pred_m": pred_m,
        "run_id": sample["run_id"],
        "sample_id": sample["sample_id"],
    }


raiz_cnn = localizar_raiz_projeto_cnn()
amostras_cnn = coletar_amostras_depth_cnn(raiz_cnn / "datasets" / "depth_ground_truth", raiz_cnn)
print(f"Amostras validas encontradas: {len(amostras_cnn)}")
print(amostras_cnn.groupby("run_id").size().rename("amostras"))

train_idx, val_idx, test_idx, split_desc = dividir_amostras_por_run(amostras_cnn)
print(split_desc)
print(f"Treino={len(train_idx)} | Validacao={len(val_idx)} | Teste={len(test_idx)}")

if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
    raise ValueError("Nao ha amostras suficientes para treinar/validar/testar a CNN de depth.")

dataset_cnn = DepthGroundTruthDataset(
    amostras_cnn,
    input_size=INPUT_SIZE_CNN,
    target_size=DEPTH_TARGET_SIZE_CNN,
    depth_max_m=DEPTH_MAX_TREINO_M,
)

train_loader_cnn = DataLoader(Subset(dataset_cnn, train_idx), batch_size=BATCH_SIZE_CNN, shuffle=True, num_workers=0)
val_loader_cnn = DataLoader(Subset(dataset_cnn, val_idx), batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=0)
test_loader_cnn = DataLoader(Subset(dataset_cnn, test_idx), batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=0)

device_cnn = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo PyTorch: {device_cnn}")

modelo_cnn_sem_atencao, hist_sem_atencao, metricas_sem_atencao = treinar_modelo_depth_cnn(
    "CNN sem atencao",
    usar_atencao=False,
    train_loader=train_loader_cnn,
    val_loader=val_loader_cnn,
    test_loader=test_loader_cnn,
    device=device_cnn,
)

modelo_cnn_com_atencao, hist_com_atencao, metricas_com_atencao = treinar_modelo_depth_cnn(
    "CNN com 1 layer de atencao",
    usar_atencao=True,
    train_loader=train_loader_cnn,
    val_loader=val_loader_cnn,
    test_loader=test_loader_cnn,
    device=device_cnn,
)

metricas_cnn = pd.DataFrame([metricas_sem_atencao, metricas_com_atencao])
display(metricas_cnn)

historico_cnn = pd.concat([hist_sem_atencao, hist_com_atencao], ignore_index=True)
fig_hist_cnn = px.line(
    historico_cnn,
    x="epoch",
    y="val_map_mae_m",
    color="modelo",
    markers=True,
    title="Validacao: MAE do mapa de depth por epoca",
    labels={"val_map_mae_m": "MAE mapa (m)", "epoch": "Epoca"},
)
fig_hist_cnn.show()

metricas_long_cnn = metricas_cnn.melt(
    id_vars="modelo",
    value_vars=["map_mae_m", "p10_mae_m", "p50_mae_m", "close5_mae_pp", "inferencia_ms_por_frame"],
    var_name="metrica",
    value_name="valor",
)
fig_metricas_cnn = px.bar(
    metricas_long_cnn,
    x="metrica",
    y="valor",
    color="modelo",
    barmode="group",
    title="Teste: CNN simples vs CNN com uma layer de atencao",
)
fig_metricas_cnn.show()

indice_exemplo = test_idx[0]
exemplo_sem = prever_um_exemplo_depth_cnn(modelo_cnn_sem_atencao, dataset_cnn, indice_exemplo, device_cnn)
exemplo_com = prever_um_exemplo_depth_cnn(modelo_cnn_com_atencao, dataset_cnn, indice_exemplo, device_cnn)

fig_exemplo_cnn = go.Figure()
fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_sem["target_m"], coloraxis="coloraxis", name="Ground truth"))
fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_sem["pred_m"], coloraxis="coloraxis", name="CNN sem atencao", visible=False))
fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_com["pred_m"], coloraxis="coloraxis", name="CNN com atencao", visible=False))
fig_exemplo_cnn.update_layout(
    title=(
        "Exemplo de mapa de depth reduzido - "
        f"{exemplo_sem['run_id']} / sample {exemplo_sem['sample_id']}"
    ),
    coloraxis={"colorscale": "Viridis", "cmin": 0, "cmax": DEPTH_MAX_TREINO_M},
    updatemenus=[{
        "buttons": [
            {"label": "Ground truth", "method": "update", "args": [{"visible": [True, False, False]}]},
            {"label": "CNN sem atencao", "method": "update", "args": [{"visible": [False, True, False]}]},
            {"label": "CNN com atencao", "method": "update", "args": [{"visible": [False, False, True]}]},
        ]
    }],
)
fig_exemplo_cnn.show()
